# E-INV — independent verification

**Run all.** CPU only, ~3 minutes. No GPU, no model weights, no training.

This notebook recomputes **every headline number in the manuscript from the per-row
measurement CSVs** and compares each against the value the paper reports. It
deliberately does **not** read the cached result JSONs — those are what it is checking.
If a stored summary has drifted from the underlying measurements, it shows up here as
`DISAGREE` rather than being silently inherited.

It also runs the **leave-flagged-out copy-audit sensitivity** automatically when the
full fine-tuning copy audit is present.

### What it verifies
| stage | quantities |
|---|---|
| Autoencoder round trip | $R_{\mathrm{real}}$, $R_{\mathrm{VAE}}$, $\eta$ with bootstrap CI, pre/post AUC |
| Primary bound | six adapter means, $U_{\mathrm{device}}$, $\lambda_U$, $\theta_{\mathrm{sym}}$, **exact** sign-flip $p$ over all 64 assignments |
| FLUX.1-dev | arm means, $\lambda_U$ at $n=3$ |
| Full fine-tuning | arm means, $\theta_{\mathrm{sym}}$, $\lambda_U$, the contaminated per-arm value, additive/interaction ratio |
| Copy audit | flag rate, and $\theta_{\mathrm{sym}}$ / $\lambda_U$ recomputed with flagged generations excluded |
| Low/mid band | arm means, symmetric bound, and the per-arm value it must not be confused with |
| Kodak, 5 devices | ICC(1,1), grand mean, device-level sign-flip $p$ |
| Additive decomposition | cross-seed $R^2$, device-level $r$, exact 120-assignment permutation $p$ |

### Reading the output
`OK` matches within tolerance · `DISAGREE` resolve before submitting ·
`skip` that CSV is not under `ROOT` · `info` no manuscript value to compare against.

Missing experiment folders are skipped, not failed, so a partial copy of the data still
verifies whatever it contains.

> Two archived JSONs hold **superseded** values by design (`E_LOWFREQ` $\lambda_U$ and the
> `XVAL` permutation $p$). This notebook recomputes both correctly, so it agrees with the
> paper and disagrees with those files. That is intended; see
> `config/results_corrections.json`.

In [ ]:
# %% 1 · mount, imports, configuration
import os, sys, json, itertools, warnings, subprocess
warnings.filterwarnings("ignore")

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DEFAULT_ROOT = "/content/drive/MyDrive/inv_channel"
except Exception:
    DEFAULT_ROOT = os.environ.get("EINV_ROOT", "./inv_channel")

# ---- point this at the folder containing E_INV_P0_v3/, E_SEEDEXT/, ... ----
ROOT = DEFAULT_ROOT

try:                                     # already present on Colab and most setups
    import numpy as np, pandas as pd
    from scipy import stats as sps
except ImportError:
    for extra in ([], ["--break-system-packages"]):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "numpy", "pandas", "scipy"] + extra,
                       check=False, capture_output=True)
    import numpy as np, pandas as pd
    from scipy import stats as sps

RNG = np.random.default_rng(0)
R_REAL_REF = 3.56703416571125e-02        # the denominator used throughout the paper

if not os.path.isdir(ROOT):
    raise SystemExit(f"ROOT is not a directory: {ROOT}\n"
                     "Set ROOT to the folder that contains E_INV_P0_v3/ etc.")
ALL = ("E_INV_P0_v3","E_SEEDEXT","E_AMP","E_TRACKB2_FLUX",
       "E_MULTIDEV","E_LOWFREQ","E_FULLFT")
present = [d for d in ALL if os.path.isdir(os.path.join(ROOT, d))]
print(f"ROOT: {ROOT}")
print(f"experiment roots found ({len(present)}/{len(ALL)}): "
      f"{', '.join(present) if present else 'NONE'}")
missing = [d for d in ALL if d not in present]
if missing: print(f"absent (their checks will report 'skip'): {', '.join(missing)}")

In [ ]:
# %% 2 · report object and shared helpers
class Report:
    def __init__(self): self.rows = []
    def add(self, name, expected, got, tol=None, note=""):
        if got is None:      self.rows.append((name, expected, None, "SKIP", note)); return
        if expected is None: self.rows.append((name, None, got, "INFO", note)); return
        rel = abs(got - expected) / max(abs(expected), 1e-30)
        ok = rel <= (tol if tol is not None else 0.01)
        self.rows.append((name, expected, got, "OK" if ok else "DISAGREE",
                          note or f"rel {rel:.2%}"))
    def show(self):
        w = max(len(r[0]) for r in self.rows) + 2
        print("\n" + "=" * (w + 60))
        print(f"{'quantity':<{w}}{'manuscript':>14}{'recomputed':>14}  {'':<11}note")
        print("-" * (w + 60))
        for n, e, g, s, note in self.rows:
            f = lambda v: "—" if v is None else (f"{v:.6g}" if abs(v) < 1e4 else f"{v:.4e}")
            mark = {"OK":"  ok","DISAGREE":"  ** DISAGREE","SKIP":"  skip","INFO":"  info"}[s]
            print(f"{n:<{w}}{f(e):>14}{f(g):>14}{mark:<13}{note}")
        print("=" * (w + 60))
        bad  = [r for r in self.rows if r[3] == "DISAGREE"]
        print(f"\n{sum(1 for r in self.rows if r[3]=='OK')} verified · "
              f"{len(bad)} disagreements · "
              f"{sum(1 for r in self.rows if r[3]=='SKIP')} skipped (data absent)")
        if bad:
            print("\nDISAGREEMENTS — resolve before submitting:")
            for r in bad: print(f"   {r[0]}: manuscript {r[1]:.6g}, recomputed {r[2]:.6g}")
        else:
            print("\nEvery quantity present in the data matches the manuscript.")
        return len(bad)

REP = Report()

def load(sub, name):
    p = os.path.join(ROOT, sub, "csv", name)
    if not os.path.exists(p): return None
    try: return pd.read_csv(p)
    except Exception as e: print(f"   ! {p}: {e}"); return None

def col(df, *cands):
    for x in cands:
        if x in df.columns: return x
    return None

def valcol(df):
    """The correlation column. Different stages named it differently, so resolve it
    rather than assuming 'rho'."""
    c = col(df, "rho", "corr", "ncc", "value", "rho_gen", "r", "score")
    if c is not None: return c
    num = [c for c in df.columns
           if df[c].dtype.kind == "f" and df[c].abs().max() <= 1.5]
    return num[-1] if num else None

def ucl(x, alpha=0.01):
    k = len(x)
    return float(np.mean(x) + float(sps.t.ppf(1-alpha/2, df=k-1))
                 * np.std(x, ddof=1) / np.sqrt(k))

def paired_theta(df, tag, own, oth, tagc="tag", kc="K", idc=None, rc=None,
                 drop=frozenset()):
    idc = idc or col(df, "gen_idx", "idx", "i")
    rc  = rc  or valcol(df)
    if idc is None or rc is None: return None
    a = df[(df[tagc]==tag) & (df[kc]==own)].sort_values(idc)
    b = df[(df[tagc]==tag) & (df[kc]==oth)].sort_values(idc)
    if not len(a) or not len(b): return None
    ids = a[idc].to_numpy(); va = a[rc].to_numpy(float); vb = b[rc].to_numpy(float)
    n = min(len(va), len(vb))
    keep = ~np.isin(ids[:n], list(drop))
    return (va[:n] - vb[:n])[keep]

def exact_signflip(m):
    """Exact one-sided sign-flip p over all 2^k assignments (no Monte Carlo)."""
    obs = float(np.mean(m))
    c = sum(1 for sg in itertools.product([-1,1], repeat=len(m))
            if float(np.mean(m*np.array(sg))) >= obs)
    return c / 2**len(m)

def arm3(d, arm, own, oth, tagc, kc, drop=frozenset()):
    """Three-seed arm mean, tolerant of the _flux / _ft / _r16 tag suffixes."""
    out = []
    for s in (0,1,2):
        for cand in (f"{arm}_ft_s{s}", f"{arm}_raw_s{s}_flux",
                     f"{arm}_raw_s{s}_r16", f"{arm}_s{s}"):
            x = paired_theta(d, cand, own, oth, tagc=tagc, kc=kc,
                             drop={g for (t,g) in drop if t == cand})
            if x is not None and len(x):
                out.append(float(x.mean())); break
    return np.array(out)

print("helpers ready")
print("\ncolumn layout of each measurement CSV found:")
for sub, nm in (("E_INV_P0_v3","s5_measure.csv"), ("E_SEEDEXT","b3_measure.csv"),
                ("E_TRACKB2_FLUX","f3_measure.csv"), ("E_FULLFT","f3_measure.csv"),
                ("E_MULTIDEV","c3_measure_raw.csv"), ("E_LOWFREQ","l2_measure.csv"),
                ("E_INV_P0_v3","s1b_vae_roundtrip.csv"), ("E_FULLFT","f5_copy.csv")):
    _d = load(sub, nm)
    if _d is None:
        print(f"  {sub}/{nm:26s} absent")
    else:
        print(f"  {sub}/{nm:26s} {list(_d.columns)}   -> value column: {valcol(_d)}")

In [ ]:
# %% 3 · autoencoder round trip — R_real, R_VAE, eta with CI, AUC
d = load("E_INV_P0_v3", "s1b_vae_roundtrip.csv")
if d is None:
    for n in ("R_real","R_VAE","eta","eta CI low","eta CI high","AUC pre","AUC post"):
        REP.add(n, None, None, note="s1b_vae_roundtrip.csv absent")
else:
    rc, sc = col(d,"role"), col(d,"stage")
    ra, rb = col(d,"rho_A","rhoA"), col(d,"rho_B","rhoB")
    ic = col(d,"idx","gen_idx")
    def con(stage):
        a = d[(d[rc]=="A") & (d[sc]==stage)].sort_values(ic)
        b = d[(d[rc]=="B") & (d[sc]==stage)].sort_values(ic)
        return (a[ra].to_numpy(float)-a[rb].to_numpy(float),
                b[rb].to_numpy(float)-b[ra].to_numpy(float))
    pa, pb = con("pre"); qa, qb = con("post")
    R_real = 0.5*(pa.mean()+pb.mean()); R_vae = 0.5*(qa.mean()+qb.mean())
    REP.add("R_real", 3.56703e-02, float(R_real))
    REP.add("R_VAE",  1.30593e-02, float(R_vae))
    REP.add("eta",    0.3661,      float(R_vae/R_real))
    et = []
    for _ in range(10000):
        ia, ib = RNG.integers(0,len(pa),len(pa)), RNG.integers(0,len(pb),len(pb))
        et.append((0.5*(qa[ia].mean()+qb[ib].mean())) /
                  (0.5*(pa[ia].mean()+pb[ib].mean())))
    lo, hi = np.percentile(et,[2.5,97.5])
    REP.add("eta CI low",  0.3437, float(lo), tol=0.03)
    REP.add("eta CI high", 0.3870, float(hi), tol=0.03)
    def auc(stage):
        a = d[(d[rc]=="A")&(d[sc]==stage)]; b = d[(d[rc]=="B")&(d[sc]==stage)]
        same  = np.r_[a[ra].to_numpy(float), b[rb].to_numpy(float)]
        cross = np.r_[a[rb].to_numpy(float), b[ra].to_numpy(float)]
        lab = np.r_[np.ones(len(same)), np.zeros(len(cross))]; s_ = np.r_[same, cross]
        o = np.argsort(s_); rk = np.empty(len(s_)); rk[o] = np.arange(len(s_))
        return float((rk[lab==1].mean()-(lab.sum()-1)/2)/(len(s_)-lab.sum()))
    REP.add("AUC pre",  1.0000, auc("pre"),  tol=0.002)
    REP.add("AUC post", 0.9814, auc("post"), tol=0.01)
    print(f"[1/6] autoencoder round trip: eta = {R_vae/R_real:.4f} "
          f"[{lo:.4f}, {hi:.4f}]")

In [ ]:
# %% 4 · primary bound (six adapters) + the exact sign-flip p-values
frames = [f for f in (load("E_INV_P0_v3","s5_measure.csv"),
                      load("E_SEEDEXT","b3_measure.csv")) if f is not None]
if not frames:
    for n in ("theta_A (n=6)","theta_B (n=6)","U_device","lambda_U plug-in %",
              "theta_sym","exact sign-flip p_A","exact sign-flip p_B"):
        REP.add(n, None, None, note="s5/b3 measure CSVs absent")
else:
    d = pd.concat(frames, ignore_index=True)
    tagc, kc = col(d,"tag"), col(d,"K")
    per = {}
    for arm, own, oth in (("A","A","B"),("B","B","A")):
        m = []
        for t in sorted(t for t in d[tagc].unique() if str(t).startswith(f"{arm}_raw_s")):
            x = paired_theta(d, t, own, oth, tagc=tagc, kc=kc)
            if x is not None and len(x): m.append(float(x.mean()))
        per[arm] = np.array(m)
    nA, nB = len(per["A"]), len(per["B"])
    print(f"[2/6] primary: {nA} adapters in arm A, {nB} in arm B")
    if nA >= 2 and nB >= 2:
        REP.add(f"theta_A (n={nA})", 1.549e-05, float(per["A"].mean()), tol=0.05)
        REP.add(f"theta_B (n={nB})", 1.438e-05, float(per["B"].mean()), tol=0.05)
        U = max(ucl(per["A"]), ucl(per["B"]))
        REP.add("U_device", 8.8802e-05, U, tol=0.05)
        REP.add("lambda_U plug-in %", 0.2490, 100*U/R_REAL_REF, tol=0.05)
        REP.add("theta_sym", 1.494e-05,
                float(0.5*(per["A"].mean()+per["B"].mean())), tol=0.05)
        if nA == 6 and nB == 6:
            REP.add("exact sign-flip p_A", 0.2969, exact_signflip(per["A"]), tol=0.10)
            REP.add("exact sign-flip p_B", 0.1875, exact_signflip(per["B"]), tol=0.10)
            REP.add("sign-flip floor 1/64", 0.015625, 1/64, tol=1e-9,
                    note="structural: cannot reject at alpha=0.01")
    else:
        REP.add("U_device", 8.8802e-05, None, note="fewer than 2 adapters per arm")

In [ ]:
# %% 5 · FLUX.1-dev, full fine-tuning, and the copy-audit sensitivity
d = load("E_TRACKB2_FLUX","f3_measure.csv")
if d is None:
    REP.add("FLUX lambda_U % (n=3)", 0.777, None, note="FLUX f3_measure.csv absent")
else:
    tagc, kc = col(d,"tag"), col(d,"K")
    A, B = arm3(d,"A","A","B",tagc,kc), arm3(d,"B","B","A",tagc,kc)
    if len(A)>=3 and len(B)>=3:
        REP.add("FLUX theta_A",  1.342e-05, float(A.mean()), tol=0.05)
        REP.add("FLUX theta_B", -7.867e-06, float(B.mean()), tol=0.05)
        REP.add("FLUX lambda_U % (n=3)", 0.777,
                100*max(ucl(A),ucl(B))/R_REAL_REF, tol=0.05)
        print(f"[3/6] FLUX: lambda_U = {100*max(ucl(A),ucl(B))/R_REAL_REF:.3f}% at n=3")
    else:
        REP.add("FLUX lambda_U % (n=3)", 0.777, None, note="fewer than 3 adapters/arm")

FT = None
d = load("E_FULLFT","f3_measure.csv")
if d is None:
    REP.add("full-FT lambda_U % (sym, n=3)", 0.7466, None,
            note="E_FULLFT f3_measure.csv absent")
else:
    tagc, kc = col(d,"tag"), col(d,"K")
    A, B = arm3(d,"A","A","B",tagc,kc), arm3(d,"B","B","A",tagc,kc)
    if len(A)>=3 and len(B)>=3:
        sym, anti = (A+B)/2, (A-B)/2
        REP.add("full-FT theta_A", -1.134e-04, float(A.mean()), tol=0.05)
        REP.add("full-FT theta_B",  6.829e-05, float(B.mean()), tol=0.05)
        REP.add("full-FT theta_sym", -2.2553e-05, float(sym.mean()), tol=0.05)
        U = ucl(sym)
        REP.add("full-FT lambda_U % (sym, n=3)", 0.7466, 100*U/R_REAL_REF, tol=0.05)
        REP.add("full-FT lambda_U % (per-arm)", 2.716,
                100*max(ucl(A),ucl(B))/R_REAL_REF, tol=0.05,
                note="additive-contaminated; NOT the reported bound")
        REP.add("full-FT |additive/interaction|", 4.03,
                float(abs(anti.mean()/sym.mean())), tol=0.05)
        FT = (d, tagc, kc, float(sym.mean()), U)
        print(f"[4/6] full FT: lambda_U = {100*U/R_REAL_REF:.4f}% (symmetric), "
              f"{100*max(ucl(A),ucl(B))/R_REAL_REF:.3f}% (per-arm, contaminated)")
    else:
        REP.add("full-FT lambda_U % (sym, n=3)", 0.7466, None,
                note="fewer than 3 adapters/arm")

# ---- leave-flagged-out sensitivity ----
f = load("E_FULLFT","f5_copy.csv")
if f is None or FT is None:
    REP.add("copy-flag rate %", 0.07, None, note="f5_copy.csv or f3_measure.csv absent")
else:
    d, tagc, kc, sym0, U0 = FT
    fc, tc_, gc_ = col(f,"copy_flag"), col(f,"tag"), col(f,"gen_idx")
    bad = {(r[tc_], int(r[gc_])) for _, r in f[f[fc]==1].iterrows()}
    REP.add("copy-flag rate %", 0.07, float(100.0*f[fc].mean()), tol=0.35,
            note=f"{len(bad)} of {len(f)} generations")
    if bad:
        A2, B2 = arm3(d,"A","A","B",tagc,kc,drop=bad), arm3(d,"B","B","A",tagc,kc,drop=bad)
        sym2 = (A2+B2)/2; U2 = ucl(sym2)
        shift = 100*abs(sym2.mean()-sym0)/abs(sym0)
        REP.add("full-FT theta_sym, flagged excluded", None, float(sym2.mean()),
                note=f"shift {shift:.2f}% of its magnitude")
        REP.add("full-FT lambda_U %, flagged excluded", None, 100*U2/R_REAL_REF,
                note=f"vs {100*U0/R_REAL_REF:.4f}% with all generations")
        print(f"\n   LEAVE-FLAGGED-OUT SENSITIVITY  (excluded: {sorted(bad)})")
        print(f"     theta_sym  {sym0:+.4e} -> {sym2.mean():+.4e}   ({shift:.2f}% shift)")
        print(f"     lambda_U   {100*U0/R_REAL_REF:.4f}% -> {100*U2/R_REAL_REF:.4f}%")
        print("     => " + ("shift is small relative to the estimate; the copy audit is closed."
                            if shift < 20 else
                            "shift is material; report the leave-flagged-out value as primary."))
    else:
        print("\n   nothing flagged by the copy audit; sensitivity is vacuous.")

In [ ]:
# %% 6 · low/mid band and the Kodak five-device experiment
d = load("E_LOWFREQ","l2_measure.csv")
gp = os.path.join(ROOT,"E_LOWFREQ","fingerprints","l0_gates.json")
if d is None:
    for n in ("low/mid theta_A","low/mid theta_B","low/mid theta_sym",
              "low/mid lambda_U % (sym)"):
        REP.add(n, None, None, note="l2_measure.csv absent")
else:
    tagc, lc = col(d,"tag"), col(d,"L","K")
    per = {}
    for arm, own, oth in (("A","A","B"),("B","B","A")):
        m = [float(x.mean()) for s in range(6)
             for x in [paired_theta(d, f"{arm}_raw_s{s}_r16", own, oth, tagc=tagc, kc=lc)]
             if x is not None and len(x)]
        per[arm] = np.array(m)
    if len(per["A"])>=2 and len(per["B"])>=2:
        REP.add("low/mid theta_A", -1.110e-04, float(per["A"].mean()), tol=0.05)
        REP.add("low/mid theta_B",  9.032e-05, float(per["B"].mean()), tol=0.05)
        sym = (per["A"]+per["B"])/2
        REP.add("low/mid theta_sym", -1.036e-05, float(sym.mean()), tol=0.10)
        if os.path.exists(gp):
            R_low = json.load(open(gp)).get("R_real_lower99")
            if R_low:
                REP.add("low/mid lambda_U % (sym)", 9.72, 100*ucl(sym)/R_low, tol=0.05)
                REP.add("low/mid lambda_U % (per-arm)", 20.56,
                        100*max(ucl(per["A"]),ucl(per["B"]))/R_low, tol=0.05,
                        note="additive-contaminated; NOT the reported bound")
        print(f"[5/6] low/mid: theta_sym = {sym.mean():+.3e}")

d = load("E_MULTIDEV","c3_measure_raw.csv")
if d is None:
    for n in ("Kodak ICC(1,1)","Kodak grand mean theta","Kodak sign-flip p (devices)",
              "additive R^2 cross-seed","additive device-level r",
              "additive exact permutation p"):
        REP.add(n, None, None, note="c3_measure_raw.csv absent")
else:
    tagc, kc = col(d,"tag"), col(d,"K")
    devs = sorted(d[kc].unique()); rows = []
    for t in sorted(d[tagc].unique()):
        dv = str(t).split("_")[0]
        if dv not in devs: continue
        _v = valcol(d); _i = col(d, "gen_idx", "idx")
        own = d[(d[tagc]==t)&(d[kc]==dv)].sort_values(_i)[_v].to_numpy(float)
        oth = [d[(d[tagc]==t)&(d[kc]==o)].sort_values(_i)[_v].to_numpy(float)
               for o in devs if o != dv]
        n = min([len(own)]+[len(x) for x in oth])
        if n: rows.append((dv, int(str(t).split("_s")[-1]),
                           float((own[:n]-np.mean([x[:n] for x in oth],axis=0)).mean())))
    if len(rows) >= 6:
        R = pd.DataFrame(rows, columns=["dev","seed","theta"])
        dm = R.groupby("dev")["theta"].mean(); gm = float(R.theta.mean())
        k, nps = len(dm), R.seed.nunique()
        ms = float(nps*((dm-gm)**2).sum()/(k-1))
        within = float(sum(((R[R.dev==dv].theta-dm[dv])**2).sum()
                           for dv in dm.index)/(k*(nps-1)))
        var_u = max((ms-within)/nps, 0.0)
        REP.add("Kodak ICC(1,1)", 0.943, float(var_u/(var_u+within)), tol=0.03)
        REP.add("Kodak grand mean theta", 1.265e-05, gm, tol=0.05)
        REP.add("Kodak sign-flip p (devices)", 0.434,
                float(exact_signflip(dm.to_numpy())), tol=0.20)

        M = d.groupby([tagc,kc])[valcol(d)].mean().unstack()
        dev_of  = {t: str(t).split("_")[0] for t in M.index}
        seed_of = {t: int(str(t).split("_s")[-1]) for t in M.index}
        DEV = list(M.columns)
        byd = {}
        for fs in sorted(set(seed_of.values())):
            tr = [t for t in M.index if seed_of[t]==fs]
            te = [t for t in M.index if seed_of[t]!=fs]
            if not tr or not te: continue
            g0 = M.loc[tr].values.mean(); b = M.loc[tr].mean(0) - g0
            for t in te:
                dv = dev_of[t]
                p_ = float(b[dv] - np.mean([b[o] for o in DEV if o != dv]))
                o_ = float(M.loc[t,dv] - np.mean([M.loc[t,o] for o in DEV if o != dv]))
                byd.setdefault(dv, []).append((p_, o_))
        if byd:
            pr = np.array([v for _, vs in sorted(byd.items()) for v,_ in vs])
            ob = np.array([v for _, vs in sorted(byd.items()) for _,v in vs])
            REP.add("additive R^2 cross-seed", 0.870,
                    float(1-((ob-pr)**2).sum()/((ob-ob.mean())**2).sum()), tol=0.08)
            Pd = np.array([np.mean([p for p,_ in v]) for _,v in sorted(byd.items())])
            Od = np.array([np.mean([o for _,o in v]) for _,v in sorted(byd.items())])
            if len(Pd) == 5:
                REP.add("additive device-level r", 0.979,
                        float(sps.pearsonr(Pd,Od)[0]), tol=0.05)
                r2f = lambda o,p: 1-((o-p)**2).sum()/((o-o.mean())**2).sum()
                obs_r2 = r2f(Od,Pd)
                null = [r2f(Od, Pd[list(pm)]) for pm in itertools.permutations(range(5))]
                REP.add("additive exact permutation p", 0.0167,
                        float(sum(1 for v in null if v>=obs_r2)/120), tol=0.45,
                        note="exact over all 120 device assignments")
        print(f"[6/6] Kodak: ICC = {var_u/(var_u+within):.3f}, "
              f"grand mean {gm:+.3e}")

In [ ]:
# %% 7 · verdict
nbad = REP.show()
print("""
Reading this table
  OK        recomputed from the per-row CSVs and matches the manuscript.
  DISAGREE  the manuscript and the raw data differ — resolve before submitting.
  skip      that CSV is not present under ROOT.
  info      reported for context; no manuscript value to compare against.

Two archived JSONs hold superseded values by design — E_LOWFREQ's lambda_U (a per-arm
bound where the symmetric one is correct) and XVAL's p_perm (a permutation that could not
reject). This notebook recomputes both correctly, so it agrees with the paper and
disagrees with those two files. See config/results_corrections.json.
""")
print("VERDICT:", "every checked quantity reproduces." if nbad == 0
      else f"{nbad} quantit{'y' if nbad==1 else 'ies'} did not reproduce — see above.")